# E3, bloque B: exportación a ONNX de las 10 redes del E2

1. Entorno de ejecución → Cambiar tipo → **GPU** (sirve una T4; no hace falta la L4).
2. Ejecutar las celdas en orden. Lee los checkpoints de `/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2` (solo lectura) y escribe en
   `/content/drive/MyDrive/tesis_imo_e3_artefactos_a0498_v2` una carpeta por red con el `.onnx` y su `metadata.json`, más `resumen_exportacion.json`.
3. Coloca también `compactas_a0498_e3.zip` en la raíz de MyDrive.
4. Al terminar, descargar la carpeta de salida completa desde Drive.

FCN/U-Net/SegNet usan a0498, cinco semillas verificadas y selección de semilla mediana.
Las otras redes son artefactos históricos E2, no las futuras diez semillas E3.

No se usa test. Una red que no exporta no detiene a las demás: queda con su error en `metadata.json`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
CORRIDAS = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')
assert CORRIDAS.is_dir(), 'No está la carpeta de corridas del E2 en Drive'
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_artefactos_a0498_v2')
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')
def sha_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'
BASE=Path('/content/imo_e2_v2')
BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name)==expected, name
PROJECT=BASE/'tesis_imo_para_codex'
print('Paquete íntegro:',len(manifest['files']),'archivos')


## Entorno aislado y fijado
Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
pip_install('transformers==4.46.3')
pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')
subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)


In [ ]:
pip_install('onnx==1.17.0', 'onnxruntime==1.20.1', 'fvcore==0.1.5.post20221221')
subprocess.run([PY, '-c', 'import onnx, onnxruntime; print(onnx.__version__, onnxruntime.__version__)'], check=True)


In [ ]:
CONTENIDOS = {'training/exportar_onnx.py': '"""Bloque B del E3: elige el checkpoint de cada red, lo exporta a ONNX y hace la prueba de ida y vuelta.\n\nCorre en Colab, en el entorno del benchmark del E2 (el `PY` del cuaderno, con torch 2.5.1 y mamba_ssm), con el\npaquete del E2 descomprimido (`--e2` = carpeta tesis_imo_para_codex) y los checkpoints en Drive.\n\nReglas, fijadas antes de mirar resultados del E3:\n- **Configuración:** FCN, U-Net y SegNet usan a0498, ganadora de la búsqueda E2,\n  con hashes de los 15 checkpoints verificados. Las demás redes siguen siendo\n  artefactos históricos E2; no representan las futuras diez semillas E3 ni el SegFormer corregido.\n- **Reentrenamiento final (p. 6):** la política, idéntica para todas las redes y para el original, es que el modelo\n  final es el checkpoint de la época elegida en validación, entrenado sobre el conjunto de entrenamiento completo\n  (las 1.015 imágenes de train). No hay un segundo entrenamiento: la validación solo eligió la época.\n- **Artefacto por red:** la semilla cuyo Dice medio por grupo en validación es la mediana de las semillas\n  disponibles (con 5 semillas, la tercera; empate: la semilla menor). Representa a la red sin elegir su mejor corrida.\n- **Prueba de ida y vuelta:** en las 218 imágenes de validación, error absoluto máximo de la probabilidad < 1e-4,\n  coincidencia de la máscara (umbral 0,5) y diferencia del Dice medio por grupo entre el ONNX y el modelo en memoria.\n- **Tamaño:** si el ONNX pasa de 95 MB (límite de GitHub, 100 MB), se cuantiza a INT8 dinámico y se mide la pérdida de\n  Dice en validación; se acepta si es menor que δ = 0,02. Si no se acepta o la exportación falla, la red queda con\n  resultados precalculados en el dashboard y el motivo en `metadata.json`.\n- **Costo:** latencia por imagen en CPU con ONNX Runtime (mediana y p95 de 50 pasadas tras 5 de calentamiento, un hilo\n  y todos los hilos), parámetros y FLOPs (fvcore, si está), memoria residente pico durante la inferencia.\n\nUso:\n  python exportar_onnx.py --e2 /content/imo_e2_v2/tesis_imo_para_codex \\\n      --corridas /content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2 --salida /content/drive/MyDrive/tesis_imo_e3_artefactos_v1 \\\n      [--modelos fcn unet ...] [--dispositivo cuda]\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nimport statistics\nimport sys\nimport time\nimport traceback\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\nTOL_PROB = 1e-4\nLIMITE_MB = 95\nDELTA = 0.02\nOPSET = 17\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1 << 20), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef dice_por_grupo(probs, filas, metrics):\n    grupos = defaultdict(list)\n    for p, (fila, y, v) in zip(probs, filas):\n        grupos[fila[\'group_id\']].append(metrics(p, y, v)[\'Dice\'])\n    return float(np.mean([np.mean(x) for x in grupos.values()]))\n\n\ndef elegir_semilla(corridas, modelo):\n    """Semilla de Dice de validación mediano entre las corridas completas del modelo."""\n    res = []\n    for r in sorted(Path(corridas).glob(f\'{modelo}_seed*/result.json\')):\n        d = json.loads(r.read_text())\n        if d.get(\'completed\'):\n            res.append((d[\'best_val_group_Dice\'], d[\'seed\'], r.parent / d[\'best_file\']))\n    if not res:\n        raise FileNotFoundError(f\'Sin corridas completas de {modelo} en {corridas}\')\n    res.sort(key=lambda t: (t[0], t[1]))\n    mediana = res[(len(res) - 1) // 2]\n    return {\'semillas_disponibles\': [s for _, s, _ in sorted(res, key=lambda t: t[1])],\n            \'val_por_semilla\': {s: v for v, s, _ in res}, \'semilla\': mediana[1], \'val_dice\': mediana[0],\n            \'checkpoint\': mediana[2]}\n\n\ndef latencia(sesion, x, hilos):\n    import onnxruntime as ort\n    nombre = sesion.get_inputs()[0].name\n    for _ in range(5):\n        sesion.run(None, {nombre: x})\n    t = []\n    for _ in range(50):\n        t0 = time.perf_counter()\n        sesion.run(None, {nombre: x})\n        t.append((time.perf_counter() - t0) * 1000)\n    t.sort()\n    return {\'hilos\': hilos, \'mediana_ms\': statistics.median(t), \'p95_ms\': t[int(0.95 * len(t)) - 1]}\n\n\ndef sesion_cpu(ruta, hilos=None):\n    import onnxruntime as ort\n    op = ort.SessionOptions()\n    if hilos:\n        op.intra_op_num_threads = hilos\n    return ort.InferenceSession(str(ruta), op, providers=[\'CPUExecutionProvider\'])\n\n\ndef exportar_uno(nombre, args, MODELS, main_logits, filas_val, metrics, meta):\n    import psutil\n    import torch\n    from torch import nn\n\n    destino = Path(args.salida) / nombre\n    destino.mkdir(parents=True, exist_ok=True)\n    meta.update({\'modelo\': nombre, \'fecha\': time.strftime(\'%Y-%m-%d %H:%M:%S\'), \'estado\': \'iniciado\'})\n    compacta = nombre in (\'fcn\', \'unet\', \'segnet\')\n    if compacta:\n        from seleccion_compactas import seleccionar, cargar\n        if not args.compactas:\n            raise ValueError(\'Las compactas requieren --compactas con los pesos a0498; no se exporta la heredada\')\n        esperados = json.loads(Path(args.hashes_compactas).read_text(encoding=\'utf-8\'))\n        eleccion = seleccionar(args.compactas, nombre, esperados)\n    else:\n        eleccion = elegir_semilla(args.corridas, nombre)\n    meta.update({k: (str(v) if k == \'checkpoint\' else v) for k, v in eleccion.items()})\n    meta[\'checkpoint_sha256\'] = sha(eleccion[\'checkpoint\'])\n\n    dispositivo = torch.device(args.dispositivo)\n    if compacta:\n        modelo = cargar(eleccion, nombre)\n    else:\n        modelo = MODELS[nombre]()\n        estado = torch.load(eleccion[\'checkpoint\'], map_location=\'cpu\', weights_only=True)\n        modelo.load_state_dict(estado[\'model\'])\n    modelo.eval().to(dispositivo)\n\n    class Probabilidad(nn.Module):\n        """Salida única del artefacto: probabilidad por píxel, sigmoid del logit principal."""\n        def __init__(self, m):\n            super().__init__()\n            self.m = m\n\n        def forward(self, x):\n            return torch.sigmoid(main_logits(self.m(x)))\n\n    envuelto = Probabilidad(modelo).eval()\n    meta[\'parametros\'] = int(sum(p.numel() for p in modelo.parameters()))\n    x0 = torch.from_numpy(filas_val[0][0][None]).to(dispositivo)\n\n    try:\n        from fvcore.nn import FlopCountAnalysis\n        meta[\'flops\'] = int(FlopCountAnalysis(envuelto, x0).unsupported_ops_warnings(False).total())\n        meta[\'flops_fuente\'] = \'fvcore (multiplicación-suma contada como 1)\'\n    except Exception as e:  # noqa: BLE001\n        meta[\'flops\'], meta[\'flops_fuente\'] = None, f\'no medido: {type(e).__name__}\'\n\n    # Salidas del modelo en memoria, imagen por imagen, con el mismo orden que el ONNX.\n    with torch.no_grad():\n        p_mem = [envuelto(torch.from_numpy(x[None]).to(dispositivo))[0, 0].float().cpu().numpy()\n                 for x, _, _ in filas_val_xyv(filas_val)]\n    meta[\'val_dice_memoria\'] = dice_por_grupo(p_mem, filas_meta(filas_val), metrics)\n\n    ruta = destino / f\'{nombre}.onnx\'\n    envuelto_cpu = envuelto.to(\'cpu\')\n    if nombre == \'segnet\':\n        from unpool_exportable import adaptar_segnet\n        # Misma red y pesos; unpool sin solapamiento expresado como ScatterElements.\n        envuelto_cpu.m = adaptar_segnet(modelo)\n    torch.onnx.export(envuelto_cpu, x0.cpu(), str(ruta), opset_version=OPSET, input_names=[\'imagen\'],\n                      output_names=[\'probabilidad\'], do_constant_folding=True, dynamo=False)\n    import onnx\n    onnx.checker.check_model(str(ruta))\n    meta[\'onnx_mb\'] = ruta.stat().st_size / 2 ** 20\n\n    def evaluar(ruta_onnx):\n        s = sesion_cpu(ruta_onnx)\n        nombre_in = s.get_inputs()[0].name\n        proc = psutil.Process()\n        rss0, pico = proc.memory_info().rss, 0\n        p = []\n        for x, _, _ in filas_val_xyv(filas_val):\n            p.append(s.run(None, {nombre_in: x[None]})[0][0, 0])\n            pico = max(pico, proc.memory_info().rss)\n        return p, (pico - rss0) / 2 ** 20, s\n\n    p_onnx, mem_mb, s = evaluar(ruta)\n    dif = max(float(np.abs(a - b).max()) for a, b in zip(p_mem, p_onnx))\n    coinc = float(np.mean([((a >= .5) == (b >= .5)).mean() for a, b in zip(p_mem, p_onnx)]))\n    meta[\'ida_y_vuelta\'] = {\'imagenes\': len(p_onnx), \'error_abs_max\': dif, \'tolerancia\': TOL_PROB,\n                            \'coincidencia_mascara\': coinc, \'val_dice_onnx\': dice_por_grupo(p_onnx, filas_meta(filas_val), metrics),\n                            \'aprueba\': dif < TOL_PROB and coinc == 1.0}\n    meta[\'ida_y_vuelta\'][\'diferencia_dice\'] = meta[\'ida_y_vuelta\'][\'val_dice_onnx\'] - meta[\'val_dice_memoria\']\n    meta[\'memoria_inferencia_mb\'] = mem_mb\n    x_np = filas_val[0][0][None]\n    meta[\'latencia_cpu\'] = [latencia(sesion_cpu(ruta, 1), x_np, 1), latencia(s, x_np, os.cpu_count())]\n    meta[\'artefacto\'] = ruta.name\n\n    if meta[\'onnx_mb\'] > LIMITE_MB:\n        from onnxruntime.quantization import QuantType, quantize_dynamic\n        ruta_q = destino / f\'{nombre}_int8.onnx\'\n        quantize_dynamic(str(ruta), str(ruta_q), weight_type=QuantType.QInt8)\n        p_q, _, sq = evaluar(ruta_q)\n        dq = dice_por_grupo(p_q, filas_meta(filas_val), metrics) - meta[\'val_dice_memoria\']\n        meta[\'int8\'] = {\'mb\': ruta_q.stat().st_size / 2 ** 20, \'diferencia_dice_val\': dq,\n                        \'acepta\': abs(dq) < DELTA and ruta_q.stat().st_size / 2 ** 20 <= LIMITE_MB,\n                        \'latencia_cpu\': latencia(sq, x_np, os.cpu_count())}\n        if meta[\'int8\'][\'acepta\']:\n            meta[\'artefacto\'] = ruta_q.name\n        else:\n            meta[\'artefacto\'] = None\n            meta[\'motivo_sin_artefacto\'] = \'No cabe en GitHub ni cuantizado sin perder δ: resultados precalculados\'\n    meta[\'artefacto_sha256\'] = sha(destino / meta[\'artefacto\']) if meta[\'artefacto\'] else None\n    meta[\'preprocesamiento\'] = {\'tamano\': 256, \'letterbox\': True, \'normalizacion\': args.norm,\n                                \'entrada\': \'float32 [1, 3, 256, 256], RGB normalizado; padding = 0\',\n                                \'salida\': \'float32 [1, 1, 256, 256], probabilidad; máscara = probabilidad >= 0,5\'}\n    meta[\'entorno\'] = {\'python\': platform.python_version(), \'torch\': torch.__version__,\n                       \'onnx\': __import__(\'onnx\').__version__, \'onnxruntime\': __import__(\'onnxruntime\').__version__,\n                       \'opset\': OPSET, \'cpu\': platform.processor() or platform.machine(), \'hilos\': os.cpu_count()}\n    meta[\'estado\'] = \'ok\' if meta[\'ida_y_vuelta\'][\'aprueba\'] else \'ida_y_vuelta_fallida\'\n    if meta[\'estado\'] != \'ok\':\n        meta[\'artefacto\'] = None\n        meta[\'artefacto_sha256\'] = None\n        meta[\'motivo_sin_artefacto\'] = \'No superó la prueba de ida y vuelta; no habilitar inferencia\'\n        if nombre == \'segnet\':\n            from respaldo_torchscript import exportar_respaldo\n            from seleccion_compactas import cargar\n            meta[\'respaldo_torchscript\'] = exportar_respaldo(cargar(eleccion, nombre), filas_val, destino, metrics)\n\n\n\ndef filas_val_xyv(filas):\n    return [(x, y, v) for x, y, v, _ in filas]\n\n\ndef filas_meta(filas):\n    return [(f, y, v) for _, y, v, f in filas]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--e2\', required=True)\n    ap.add_argument(\'--corridas\', required=True)\n    ap.add_argument(\'--salida\', required=True)\n    ap.add_argument(\'--modelos\', nargs=\'+\')\n    ap.add_argument(\'--dispositivo\', default=\'cpu\')\n    ap.add_argument(\'--compactas\', help=\'Carpeta que contiene fase2/*_a0498_s*/best.pt\')\n    ap.add_argument(\'--hashes-compactas\', default=str(Path(__file__).resolve().parents[1] / \'config/hashes_compactas.json\'))\n    ap.add_argument(\'--data-root\', help=\'Raíz que contiene VOC2012, sin modificar datos\')\n    ap.add_argument(\'--threads\', type=int, default=2)\n    args = ap.parse_args()\n    proyecto = Path(args.e2)\n    sys.path.insert(0, str(proyecto / \'scripts\'))\n    from dataset_ready import load_example, metrics, partition\n    import torch\n    torch.set_num_threads(args.threads)\n    if args.modelos and set(args.modelos) <= {\'fcn\', \'unet\', \'segnet\'}:\n        MODELS_E2 = {}\n        def main_logits(salida):\n            return salida\n    else:\n        from modelos_e2 import MODELS_E2, main_logits\n    args.norm = json.loads((proyecto / \'resultados/preentrenamiento/normalization_train.json\').read_text())\n    filas = []\n    for fila in sorted(partition(\'val\'), key=lambda r: r[\'id\']):\n        x, y, v = load_example(fila, data_root=args.data_root)\n        filas.append((x, y, v, fila))\n    salida = Path(args.salida)\n    salida.mkdir(parents=True, exist_ok=True)\n    resumen = {}\n    for nombre in args.modelos or list(MODELS_E2):\n        print(\'==\', nombre, flush=True)\n        meta = {}\n        try:\n            exportar_uno(nombre, args, MODELS_E2, main_logits, filas, metrics, meta)\n        except Exception as e:  # noqa: BLE001  una red que no exporta no detiene a las demás\n            # Se conserva lo ya medido (semilla, parámetros, Dice en memoria) y se agrega el error.\n            meta.update({\'estado\': \'fallo\', \'error\': f\'{type(e).__name__}: {e}\', \'traza\': traceback.format_exc()[-2000:],\n                         \'artefacto\': None, \'artefacto_sha256\': None,\n                         \'motivo_sin_artefacto\': \'Exportación fallida: resultados precalculados\'})\n        (salida / nombre).mkdir(exist_ok=True)\n        (salida / nombre / \'metadata.json\').write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n        resumen[nombre] = {k: meta.get(k) for k in [\'estado\', \'semilla\', \'onnx_mb\', \'artefacto\', \'error\']}\n        resumen[nombre][\'ida_y_vuelta\'] = (meta.get(\'ida_y_vuelta\') or {}).get(\'error_abs_max\')\n        print(json.dumps(resumen[nombre], ensure_ascii=False, default=str), flush=True)\n    (salida / \'resumen_exportacion.json\').write_text(json.dumps(resumen, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/compactas_e2.py': '"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\ndef block(a, b):\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\n\ndef _drop(activo, p):\n    return nn.Dropout(p) if activo else nn.Identity()\n\nclass FCNBusqueda(nn.Module):\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\n\n    def __init__(self, w, b, p=0.2):\n        super().__init__()\n        w1, w2, w3 = w\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\n        self.head = nn.Conv2d(w3, 1, 1)\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\n\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\'bilinear\', align_corners=False)\n\nclass UNetBusqueda(nn.Module):\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\n\n    def __init__(self, w, b, p=0.2):\n        super().__init__()\n        w1, w2, w3 = w\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\n        self.head = nn.Conv2d(w1, 1, 1)\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\n\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\'nearest\'), m], 1)))\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\'nearest\'), a], 1)))\n        return self.head(d)\n\nclass SegNetBusqueda(nn.Module):\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\n\n    def __init__(self, w, b, p=0.2):\n        super().__init__()\n        w1, w2, w3 = w\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\n        self.reduce = nn.Conv2d(w3, w2, 1)\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\n        self.head = nn.Conv2d(w1, 1, 1)\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\n\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\n        m = self.r2(self.e2(p))\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\n        c = self.reduce(self.r3(self.e3(p)))\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\n        return self.head(d)\n\nCLASES = {\'fcn\': FCNBusqueda, \'unet\': UNetBusqueda, \'segnet\': SegNetBusqueda}\n\ndef construir_red(familia, w, b, p):\n    return CLASES[familia](tuple(w), tuple(b), p)\n', 'training/seleccion_compactas.py': '"""Selección por validación y carga estricta de las compactas a0498 del E2."""\nimport hashlib\nimport json\nimport statistics\nfrom pathlib import Path\n\nFAMILIAS = (\'fcn\', \'unet\', \'segnet\')\nSEMILLAS = (11, 23, 37, 53, 71)\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef seleccionar(raiz, familia, esperado):\n    if familia not in FAMILIAS:\n        raise ValueError(f\'Familia desconocida: {familia}\')\n    raiz = Path(raiz)\n    registros = []\n    for seed in SEMILLAS:\n        nombre = f\'{familia}_a0498_s{seed}\'\n        carpeta = raiz / \'fase2\' / nombre\n        resultado = json.loads((carpeta / \'result.json\').read_text(encoding=\'utf-8\'))\n        config = json.loads((carpeta / \'config.json\').read_text(encoding=\'utf-8\'))\n        tarea = config[\'tarea\']\n        if (not resultado[\'completed\'] or resultado[\'id\'] != nombre\n                or tarea[\'familia\'] != familia or tarea[\'arq\'] != \'a0498\'\n                or tarea[\'semilla\'] != seed or tarea[\'px\'] != 256\n                or tarea[\'w\'] != [64, 64, 64] or tarea[\'b\'] != [0, 1, 0]\n                or tarea[\'dropout_p\'] != 0.2\n                or config[\'fingerprint\'] != resultado[\'fingerprint\']):\n            raise ValueError(f\'Configuración inesperada: {nombre}\')\n        peso = carpeta / \'best.pt\'\n        if sha(peso) != esperado[nombre] or resultado[\'best_sha256\'] != esperado[nombre]:\n            raise ValueError(f\'Peso alterado: {nombre}\')\n        registros.append({\'seed\': seed, \'val\': resultado[\'best_val_group_Dice\'],\n                          \'checkpoint\': peso, \'config\': config, \'result\': resultado})\n    orden = sorted(registros, key=lambda r: (r[\'val\'], r[\'seed\']))\n    valor_mediano = orden[len(orden) // 2][\'val\']\n    elegida = min((r for r in registros if r[\'val\'] == valor_mediano), key=lambda r: r[\'seed\'])\n    return {\'semilla\': elegida[\'seed\'], \'checkpoint\': elegida[\'checkpoint\'],\n            \'val_dice\': elegida[\'val\'], \'semillas_disponibles\': list(SEMILLAS),\n            \'val_por_semilla\': {r[\'seed\']: r[\'val\'] for r in registros},\n            \'configuracion\': elegida[\'config\'], \'fingerprint\': elegida[\'result\'][\'fingerprint\'],\n            \'media_val_configuracion\': statistics.mean(r[\'val\'] for r in registros),\n            \'de_val_configuracion\': statistics.stdev(r[\'val\'] for r in registros),\n            \'arquitectura\': \'a0498\', \'criterio\': \'arquitectura ganadora E2; semilla mediana por Dice val\'}\n\n\ndef cargar(eleccion, familia):\n    import torch\n    from compactas_e2 import construir_red\n    tarea = eleccion[\'configuracion\'][\'tarea\']\n    modelo = construir_red(familia, tarea[\'w\'], tarea[\'b\'], tarea[\'dropout_p\'])\n    estado = torch.load(eleccion[\'checkpoint\'], map_location=\'cpu\', weights_only=True)\n    if estado[\'fingerprint\'] != eleccion[\'fingerprint\']:\n        raise ValueError(\'Fingerprint del checkpoint diferente\')\n    modelo.load_state_dict(estado[\'model\'], strict=True)\n    return modelo.eval()\n', 'training/unpool_exportable.py': '"""Equivalencia de max_unpool2d para pools 2x2, stride 2, sin índices repetidos."""\nimport copy\nimport torch\nfrom compactas_e2 import SegNetBusqueda\nfrom torch.nn import functional as F\n\n\ndef unpool(x, indices, referencia):\n    # Los índices de MaxPool son offsets espaciales dentro de cada canal.\n    base = torch.zeros_like(referencia).flatten(2)\n    return base.scatter(2, indices.flatten(2), x.flatten(2)).reshape_as(referencia)\n\n\nclass SegNetExportable(SegNetBusqueda):\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\n        m = self.r2(self.e2(p))\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\n        c = self.reduce(self.r3(self.e3(p)))\n        d = self.s2(self.d2(unpool(c, i2, m)))\n        d = self.s1(self.d1(unpool(d, i1, a)))\n        return self.head(d)\n\n\ndef adaptar_segnet(modelo):\n    nuevo = copy.deepcopy(modelo)\n    nuevo.__class__ = SegNetExportable\n    return nuevo.eval()\n', 'training/respaldo_torchscript.py': '"""Respaldo CPU verificado cuando ONNX no cumple la tolerancia. No habilita ONNX en la app."""\nimport hashlib\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef exportar_respaldo(modelo, filas, destino, metrics):\n    from exportar_onnx import dice_por_grupo, filas_meta\n\n    class Probabilidad(torch.nn.Module):\n        def __init__(self, red):\n            super().__init__()\n            self.red = red\n\n        def forward(self, x):\n            return torch.sigmoid(self.red(x))\n\n    modelo = Probabilidad(modelo.cpu().eval()).eval()\n    ruta = Path(destino) / \'segnet_torchscript.pt\'\n    x0 = torch.from_numpy(filas[0][0][None])\n    script = torch.jit.trace(modelo, x0, check_trace=True)\n    torch.jit.save(script, str(ruta))\n    restaurado = torch.jit.load(str(ruta), map_location=\'cpu\').eval()\n    originales, exportadas = [], []\n    error, iguales = 0.0, True\n    inicio = time.perf_counter()\n    with torch.no_grad():\n        for i, (x, _, _, _) in enumerate(filas):\n            tensor = torch.from_numpy(x[None])\n            a, b = modelo(tensor).numpy()[0, 0], restaurado(tensor).numpy()[0, 0]\n            error = max(error, float(np.abs(a - b).max()))\n            iguales = iguales and np.array_equal(a >= 0.5, b >= 0.5)\n            originales.append(a)\n            exportadas.append(b)\n            if (i + 1) % 50 == 0:\n                print(f\'TorchScript: {i + 1}/{len(filas)} imágenes\', flush=True)\n    dice_a = dice_por_grupo(originales, filas_meta(filas), metrics)\n    dice_b = dice_por_grupo(exportadas, filas_meta(filas), metrics)\n    return {\'archivo\': ruta.name, \'sha256\': hashlib.sha256(ruta.read_bytes()).hexdigest(),\n            \'formato\': \'TorchScript\', \'imagenes\': len(filas), \'error_abs_max\': error,\n            \'mascaras_iguales\': bool(iguales), \'diferencia_dice\': dice_b - dice_a,\n            \'aprueba\': bool(error < 1e-4 and iguales), \'segundos_verificacion\': time.perf_counter() - inicio,\n            \'uso\': \'CPU con PyTorch; no habilitado en el dashboard ONNX ni probado en Render\'}\n', 'config/hashes_compactas.json': '{\n  "fcn_a0498_s11": "ef655608d580e34a72966f6f352e643544c3eb8c215dace5d246da98f72814c7",\n  "fcn_a0498_s23": "c9f9856e9239d9c59366871525b870ac421ef6ae370bb60b29bff6960aeb2772",\n  "fcn_a0498_s37": "f5c326a78e22f4d8dca84cdfd0b86a9cd9646e947ab22c59465fd82d4af00df4",\n  "fcn_a0498_s53": "6f220654d54344f5b6545087d3c102c0415be2d97eed8721b62ebb36789a86fa",\n  "fcn_a0498_s71": "9e2b5d911b7722573885eeb157cf75f43b5302d11eabcb1ad287f6c16ff1164e",\n  "segnet_a0498_s11": "c2bc57be34caa5501118601b9d050be50062f5a657309850e47e2a96b5ac2d14",\n  "segnet_a0498_s23": "a5d63031449a429343961f982ef26fdcf6817c139fb1c7142a3b0fd4199d2dac",\n  "segnet_a0498_s37": "7c358e59dd5ef539375b43d78a30b5f46de262418afac998a956bea0bc75e750",\n  "segnet_a0498_s53": "3c41962731ca6d1680a879f6d25608abcda7790c6e24e91ed68835463bc1376c",\n  "segnet_a0498_s71": "c18f29d092e404dc42210bf284fb4b998c02eebda495322ac42e9eb2c8518ecd",\n  "unet_a0498_s11": "4138b06d20bd32a422a69970252187322a35fea09a4e304bcda61d904ac672a9",\n  "unet_a0498_s23": "e0452e0bf9df896ffaafbbc1838c02abc070d1b37478121fc25246a33aec75aa",\n  "unet_a0498_s37": "59c6df9ad6c05500114d0e5e4948e8fd4cc5bd831eda4c577aab6066be25ddeb",\n  "unet_a0498_s53": "615d2b1a786ec37372e8ca2868783ec99ce135c556b63610b9a7bfc1e4a10f02",\n  "unet_a0498_s71": "a0d80416558ccdfd1632ad34ef1bd132ac896f2c83ab7b8b7c2f5d0adb03df84"\n}', 'config/fuentes_compactas.json': '{\n  "sources": {\n    "libro-imo-pages\\\\e2\\\\tesis_imo_para_codex\\\\scripts\\\\busqueda_arquitectura_e2.py": "8e1928be1532f82e3e21f4e8757e93dbc551a7d4f7042b32bc295948e48ad68a",\n    "libro-imo-pages\\\\e2\\\\tesis_imo_para_codex\\\\scripts\\\\modelos_base.py": "a342f30c03a12975e14c923d10362f4369ba9a9eb8dd8ad3e054e0e47e6603b2"\n  },\n  "method": "AST source segments copied verbatim; no model changes"\n}'}
FIRMAS = {'training/exportar_onnx.py': '8a8cb076bb95d6cbe927f907c25d42d96d65e9d04e3d64a623b651de24aa1530', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'training/seleccion_compactas.py': 'b1bb977f59301c08706de687a85f4404dde819fa8bccd56149f2dd9b51f418af', 'training/unpool_exportable.py': '910915a5c041db98a81dc68863d03f46f88d7819f692b78f56c0e521edba76c9', 'training/respaldo_torchscript.py': '9335abf5bf3b53cedf2a4c0e9134dbb1bceb9b891bccdfa1670709884506ef44', 'config/hashes_compactas.json': '40cc81401f13ac5bde28bcd914ca170dc6180f3b3ac9e233f70b50bbd0b6fb21', 'config/fuentes_compactas.json': '084479efa02d909230c485c7b7452e5915c05d7d01f8f4b5c30eab64d95649b0'}
E3_EXPORT = Path('/content/e3_export_a0498')
for rel, texto in CONTENIDOS.items():
    destino = E3_EXPORT / rel
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(texto, encoding='utf-8')
    assert hashlib.sha256(destino.read_bytes()).hexdigest() == FIRMAS[rel]
SCRIPT = E3_EXPORT / 'training/exportar_onnx.py'
ZIP_COMPACTAS = Path('/content/drive/MyDrive/compactas_a0498_e3.zip')
assert ZIP_COMPACTAS.is_file(), 'Sube compactas_a0498_e3.zip a la raiz de MyDrive'
COMPACTAS = Path('/content/compactas_a0498')
esperados = json.loads(CONTENIDOS['config/hashes_compactas.json'])
permitidos = {'MANIFIESTO.json'} | {f'fase2/{r}/{n}' for r in esperados for n in ('best.pt','config.json','result.json')}
with zipfile.ZipFile(ZIP_COMPACTAS) as z:
    assert len(z.namelist()) == len(set(z.namelist())) and set(z.namelist()) == permitidos
    for corrida, firma in esperados.items():
        assert hashlib.sha256(z.read(f'fase2/{corrida}/best.pt')).hexdigest() == firma
    z.extractall(COMPACTAS)  # Todos los nombres se validaron contra una lista cerrada.
print('15 pesos a0498 verificados')


## Exportar
Tarda de 20 a 60 minutos. Si se cae la sesión, repetir todas las celdas: las redes ya exportadas se vuelven a exportar (no hay reanudación parcial).


In [ ]:
subprocess.run([PY, str(SCRIPT), '--e2', str(PROJECT), '--corridas', str(CORRIDAS),
                '--salida', str(OUTPUT), '--compactas', str(COMPACTAS), '--dispositivo', 'cuda'], check=True)
print((OUTPUT / 'resumen_exportacion.json').read_text())
